# Using AutoModel

In [1]:
%pip install --upgrade transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 114.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 104.6 MB/s eta 0:00:0000:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import necessary libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import warnings
warnings.filterwarnings("ignore")

In [4]:
# Step 1: Set the random seed for reproducibility
# HINT: Use torch.random.manual_seed()
torch.random.manual_seed(34)

In [8]:
# Step 2: Load the model from Hugging Face
# HINT: Use AutoModelForCausalLM.from_pretrained()
# MODEL_NAME: "microsoft/Phi-3-mini-4k-instruct"
model = AutoModelForCausalLM.from_pretrained( 
    "microsoft/Phi-3-mini-4k-instruct", 
    device_map="cuda", 
    torch_dtype="auto",  
) 

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [9]:
# Step 3: Load the tokenizer for the model
# HINT: Use AutoTokenizer.from_pretrained()
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct") 

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [10]:
# Step 4: Define the messages for the conversation
# HINT: Create a list of dictionaries representing conversation history.
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "Can you provide ways to eat combinations of bananas and dragonfruits?"},
    {"role": "assistant", "content": "Sure! Here are some ways to eat bananas and dragonfruits together: 1. Banana and dragonfruit smoothie: Blend bananas and dragonfruits together with some milk and honey. 2. Banana and dragonfruit salad: Mix sliced bananas and dragonfruits together with some lemon juice and honey."},
    {"role": "user", "content": "What about solving a 2x + 3 = 7 equation?"},
]

In [11]:
# Step 5: Create a pipeline for text generation
# HINT: Use pipeline() with appropriate parameters for text-generation
pipe = pipeline( 
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
) 

In [12]:
# Step 6: Define generation arguments
# HINT: Set parameters like max_new_tokens, temperature, etc.
generation_args = { 
    "max_new_tokens": 500, 
    "return_full_text": False, 
    "temperature": 0.0, 
    "do_sample": False, 
} 


In [13]:
# Step 7: Check how tokenizer works
# HINT: use tokenizer
tokenizer("What about solving a 2x + 3 = 7 equation?")

{'input_ids': [1724, 1048, 17069, 263, 29871, 29906, 29916, 718, 29871, 29941, 353, 29871, 29955, 6306, 29973], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [38]:
# Step 8: Generate and print output using the pipeline
# HINT: Pass the messages and generation_args to the pipeline.
def predict_output(input_prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that assists users to find the correct methods/approach for security within an organization."},
        {"role": "user", "content": input_prompt}  # The user's input prompt is added here
    ]
    
    text = tokenizer.apply_chat_template(
        messages,  # The structured message for the assistant
        tokenize=False,  # Don't tokenize yet, just prepare the chat template
        add_generation_prompt=True 
    )
    print(text)
    
    model_inputs = tokenizer([text], return_tensors="pt")
    model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}
    print(model_inputs)
    
    generated_ids = model.generate(**model_inputs, max_new_tokens=4096,temperature=0.2)

    # Slice the generated sequence to remove the original input tokens (we only want the output)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs["input_ids"], generated_ids)]
    print(generated_ids)
    # Decode the generated token IDs back into a human-readable string, skipping special tokens
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(response)
    
    return response

In [39]:
predict_output("Ehe te nandayo")

<|system|>
You are a helpful assistant that assists users to find the correct methods/approach for security within an organization.<|end|>
<|user|>
Ehe te nandayo<|end|>
<|assistant|>

{'input_ids': tensor([[32006,   887,   526,   263,  8444, 20255,   393,  1223,  2879,  4160,
           304,  1284,   278,  1959,  3519, 29914,  9961,   496,   363,  6993,
          2629,   385, 13013, 29889, 32007, 32010, 24180,   734,   302,   392,
           388, 29877, 32007, 32001]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}
[tensor([  910, 15278,  5692,   304,   367,   297,  3925,   801,  2638, 29892,
          607,  5578,  1078,   304,   376, 29902,   626,  2675,   304,   664,
         1213,   512,   278,  3030,   310,  6993,  2629,   385, 13013, 29892,
          445,  1033, 22366,   393,   278,  5375,   338,  1048,   304,  3033,
          482,   297, 14188,  4475, 

'This instruction appears to be in Swahili, which translates to "I am going to work." In the context of security within an organization, this could imply that the individual is about to engage in activities related to their role in maintaining or enhancing security measures.\n\n\nTo address this in a security context, the individual should ensure they are following all protocols for secure access to the workplace, such as using secure passwords, two-factor authentication, and ensuring that any sensitive information is handled according to the organization\'s data protection policies. They should also be aware of their surroundings and report any suspicious activities to the appropriate security personnel.'

In [ ]:
# Add another conversation and test the model with different input.
# HINT: Change the user input in the messages and call the pipeline again.

# Using pipeline

In [47]:
import torch
import gc

# Delete unnecessary tensors
# Collect garbage
gc.collect()
# Clear cache
torch.cuda.empty_cache()


In [49]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("snehalyelmati/mt5-hindi-to-english")
model = AutoModelForSeq2SeqLM.from_pretrained("snehalyelmati/mt5-hindi-to-english")

input_text = "मैं कल तुमसे मिलने आऊंगा"
tokenized_text = tokenizer(input_text,return_tensors="pt")
translated = model.generate(**tokenized_text)
translated_text = tokenizer.batch_decode(translated, skip_special_tokens=True)[0]
print(translated_text)

Loading weights:   0%|          | 0/191 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


I'll be coming to see you tomorrow.
